# Docker, paso a paso — con el ejemplo GraphQL de la Sesión 2

Tutorial de la **Sesión 2** de MLOps II. Aprendemos **qué es Docker** y **cómo usarlo** para empaquetar y correr nuestra API GraphQL de forma reproducible.

### ¿Por qué Docker en MLOps?
Un modelo/servicio "anda en mi máquina" pero falla en otra por versiones distintas. Docker empaqueta **código + dependencias + sistema** en una **imagen** que corre igual en cualquier lado. Es la base del **nivel 'contenedores'** del TP integrador.

* **Imagen:** la plantilla inmutable (lo que construyes con `docker build`).
* **Contenedor:** una instancia en ejecución de una imagen (lo que corres con `docker run`).
* **Dockerfile:** la receta para construir la imagen.

## 1. Requisitos

* **Docker Desktop** instalado y corriendo.
* Verifica que Docker responde:

In [ ]:
!docker --version

## 2. La app a contenerizar

Escribimos una API GraphQL mínima (libros) con FastAPI + Strawberry. Es la misma idea del tutorial `GraphQL.ipynb`.

In [ ]:
%%writefile app.py
import strawberry
from typing import List, Optional
from fastapi import FastAPI
from strawberry.fastapi import GraphQLRouter

books = [
    {"id": "b1", "title": "El principito", "author": "Saint-Exupery"},
    {"id": "b2", "title": "1984", "author": "Orwell"},
]

@strawberry.type
class Book:
    id: str
    title: str
    author: str

@strawberry.type
class Query:
    @strawberry.field
    def books(self) -> List[Book]:
        return [Book(**b) for b in books]

schema = strawberry.Schema(query=Query)
app = FastAPI(title="GraphQL en Docker")
app.include_router(GraphQLRouter(schema), prefix="/graphql")

@app.get("/status")
def status():
    return {"status": "ok"}


## 3. Las dependencias

Un `requirements.txt` con lo que la imagen necesita.

In [ ]:
%%writefile requirements.txt
strawberry-graphql
fastapi
uvicorn[standard]


## 4. El Dockerfile (con uv)

La receta de la imagen. Usamos **uv** (el gestor del curso) para instalar dependencias rápido.

* `FROM` — imagen base con Python.
* `COPY --from=...uv` — trae el binario de uv.
* `RUN uv pip install` — instala las dependencias (capa cacheada).
* `COPY . .` — copia el código.
* `EXPOSE` / `CMD` — puerto y comando de arranque.

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim
COPY --from=ghcr.io/astral-sh/uv:latest /uv /bin/
WORKDIR /app
COPY requirements.txt .
RUN uv pip install --system -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]


## 5. Construir la imagen

`docker build` lee el Dockerfile y arma la imagen `graphql-app`. El `.` es el contexto (la carpeta actual).

In [ ]:
!docker build -t graphql-app .

## 6. Correr el contenedor

`-d` en segundo plano, `-p 8000:8000` mapea el puerto del contenedor al de tu máquina, `--name` le pone nombre.

In [ ]:
!docker run -d --name graphql-server -p 8000:8000 graphql-app

### Ver qué está corriendo y sus logs

In [ ]:
!docker ps

In [ ]:
!docker logs graphql-server

## 7. Probar la API dentro del contenedor

Desde tu máquina, apuntamos a `localhost:8000` (el puerto que mapeamos).

In [ ]:
import requests
q = "{ books { title author } }"
r = requests.post("http://localhost:8000/graphql", json={"query": q})
print(r.status_code, r.json())

También puedes abrir **GraphiQL** en <http://localhost:8000/graphql>.

## 8. Frenar y limpiar

Un contenedor sigue vivo hasta que lo frenas. Buenas prácticas de higiene:

In [ ]:
!docker stop graphql-server

In [ ]:
!docker rm graphql-server

Para borrar también la imagen: `!docker rmi graphql-app`.

## Cheatsheet

| Comando | Qué hace |
|---|---|
| `docker build -t nombre .` | Construir la imagen desde el Dockerfile |
| `docker run -d -p 8000:8000 --name x nombre` | Correr un contenedor |
| `docker ps` / `docker ps -a` | Contenedores corriendo / todos |
| `docker logs x` | Ver los logs |
| `docker stop x` / `docker rm x` | Frenar / borrar el contenedor |
| `docker images` / `docker rmi img` | Listar / borrar imágenes |

### Conexión con el curso
Este es el patrón del **nivel 'contenedores'** del TP integrador y de servicios como Neo4j, MLflow, MinIO y Airflow (que se levantan como contenedores). Lo profundizamos en la Sesión 5 (nube/Data Lakes) con Docker Compose.